# Streaming Process Mining with Gemini AI

This notebook demonstrates the complete streaming process mining pipeline using Google Gemini AI:
1. Load XES event log
2. Process events through sliding window and decay list
3. Prepare dataset for mining
4. Discover process model using Gemini AI (instead of pm4py)
5. Analyze and visualize results



In [3]:
# Install dependencies
!pip -q install google-generativeai




[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: C:\Users\ITWORK\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [21]:
import sys
import os
from pathlib import Path
from datetime import datetime, timedelta
from pytz import UTC
import json

# Find project root: go up from notebooks/ directory
current_dir = Path.cwd()
if current_dir.name == "notebooks":
    project_root = current_dir.parent
else:
    # Search for src/stream_mining/ directory
    project_root = current_dir
    for parent in [current_dir] + list(current_dir.parents):
        if (parent / "src" / "stream_mining").exists():
            project_root = parent
            break

# Convert to absolute path
project_root = project_root.resolve()
src_path = project_root / "src"

# Add src/ to sys.path (not src/stream_mining!)
# This allows: from stream_mining.io_xes import read_xes
src_path_str = str(src_path)
if src_path_str not in sys.path:
    sys.path.insert(0, src_path_str)

print(f"Project root: {project_root}")
print(f"Source path: {src_path}")
print(f"Added to sys.path: {src_path_str}")
print(f"sys.path[0]: {sys.path[0]}")

# Verify paths exist
if not src_path.exists():
    raise FileNotFoundError(f"Source directory not found: {src_path}")
if not (src_path / "stream_mining").exists():
    raise FileNotFoundError(f"stream_mining module not found in: {src_path}")

# Import stream mining modules
from stream_mining.io_xes import read_xes
from stream_mining.sliding_window import SlidingWindowCount
from stream_mining.decay_list import DecayList
from stream_mining.prepare_dataset import prepare_for_mining
from stream_mining.export import export_xes

# Import Gemini AI
try:
    import google.generativeai as genai
    print("✓ google-generativeai imported successfully!")
except ImportError:
    print("⚠ google-generativeai not installed. Please run: pip install google-generativeai")
    raise

print("✓ All imports successful!")



Project root: C:\github\process-mining
Source path: C:\github\process-mining\src
Added to sys.path: C:\github\process-mining\src
sys.path[0]: C:\Program Files\JetBrains\PyCharm 2023.3.5\plugins\python\helpers-pro\jupyter_debug
✓ google-generativeai imported successfully!
✓ All imports successful!


 ## Data Loading

Load the sample XES file. If running in Colab, you may need to upload the file or use a different path.



In [12]:
data_dir = project_root / "data" / "in"
xes_file = data_dir / "sample_small.xes"

if not xes_file.exists():
    xes_file = Path("sample_small.xes")
    if not xes_file.exists():
        xes_file = Path("data") / "sample_small.xes"
        if not xes_file.exists():
            print("Warning: sample_small.xes not found. Please ensure the file exists.")
            print(f"Looking for: {xes_file}")

print(f"Loading XES file: {xes_file}")
log = read_xes(str(xes_file))

print(f"\nLoaded {len(log.traces)} traces")
total_events = sum(len(trace.events) for trace in log.traces)
print(f"Total events: {total_events}")

if log.traces:
    print(f"\nFirst trace (case_id: {log.traces[0].case_id}):")
    for event in log.traces[0].events:
        print(f"  - {event.activity} at {event.timestamp}")



Loading XES file: C:\github\process-mining\data\in\sample_small.xes


parsing log, completed traces ::   0%|          | 0/3 [00:00<?, ?it/s]


Loaded 3 traces
Total events: 9

First trace (case_id: case1):
  - start at 2024-01-01 10:00:00+00:00
  - process at 2024-01-01 10:05:00+00:00
  - end at 2024-01-01 10:10:00+00:00


## Streaming Simulation

Simulate streaming processing by iterating through events and feeding them to:
1. **Sliding Window**: Maintains a fixed-size window of recent events
2. **Decay List**: Maintains historical events with exponential decay weights



In [22]:
# Initialize sliding window and decay list
window_size = 50  # Keep last 50 events in window
half_life = timedelta(hours=24)  # Events decay with 24-hour half-life

window = SlidingWindowCount(max_events=window_size)
decay = DecayList(half_life=half_life)

# Process all events from the log
print("Processing events through sliding window and decay list...")
print("Events are added to decay list ONLY when evicted from window (when window is full).")
evicted_count = 0

now = datetime.now(UTC)

for trace in log.traces:
    for event in trace.events:
        # Push to sliding window (may evict oldest event when window is full)
        evicted = window.push(event)
        
        # Only add to decay list when window is full and event is evicted
        # This is the ONLY way events enter the decay list
        if evicted:
            decay.update(evicted)
            evicted_count += 1

print(f"\nProcessing complete!")
print(f"Window size: {len(window)} events")
print(f"Decay list size: {decay.stats()['size']} events")
print(f"Events evicted from window: {evicted_count}")

# Display window stats
window_stats = window.stats()
print(f"\nWindow stats:")
print(f"  - Size: {window_stats['size']}")
if window_stats.get('oldest_timestamp'):
    print(f"  - Oldest: {window_stats['oldest_timestamp']}")
    print(f"  - Newest: {window_stats['newest_timestamp']}")

# Display decay list stats
decay_stats = decay.stats(reference_time=now)
print(f"\nDecay list stats:")
print(f"  - Size: {decay_stats['size']}")
if decay_stats.get('min_weight') is not None:
    print(f"  - Min weight: {decay_stats['min_weight']:.4f}")
    print(f"  - Max weight: {decay_stats['max_weight']:.4f}")



Processing events through sliding window and decay list...
Events are added to decay list ONLY when evicted from window (when window is full).

Processing complete!
Window size: 9 events
Decay list size: 0 events
Events evicted from window: 0

Window stats:
  - Size: 9
  - Oldest: 2024-01-01 10:00:00+00:00
  - Newest: 2024-01-01 12:15:00+00:00

Decay list stats:
  - Size: 0


## Dataset Preparation

Prepare dataset for mining:
- **Sliding window** is the PRIMARY source (all events currently in window)
- **Decay list** is used to "revive" traces that are NOT in the window
  (when a new event arrives for a case that was previously evicted)



In [14]:
event_log = prepare_for_mining(window, decay, now)

print(f"Prepared EventLog:")
print(f"  - Traces: {len(event_log)}")
total_events = sum(len(trace) for trace in event_log)
print(f"  - Total events: {total_events}")

if len(event_log) > 0:
    print(f"\nFirst few traces:")
    for i, trace in enumerate(event_log[:3]):
        case_id = trace.attributes.get("concept:name", f"trace_{i}")
        print(f"  - {case_id}: {len(trace)} events")
        if len(trace) > 0:
            activities = [event["concept:name"] for event in trace]
            print(f"    Activities: {' -> '.join(activities)}")



Prepared EventLog:
  - Traces: 3
  - Total events: 9

First few traces:
  - case1: 3 events
    Activities: start -> process -> end
  - case2: 2 events
    Activities: start -> end
  - case3: 4 events
    Activities: start -> process -> review -> end


## Gemini API Setup

Configure Gemini API using the GEMINI_API_KEY environment variable.



In [23]:
# Get API key from environment variable
api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    raise ValueError(
        "GEMINI_API_KEY environment variable not set. "
        "Please set it using: export GEMINI_API_KEY='your-api-key'"
    )

# Configure Gemini
genai.configure(api_key=api_key)

# List available models
print("Checking available Gemini models...")
try:
    available_models = [m.name for m in genai.list_models() if 'generateContent' in m.supported_generation_methods]
    print(f"Available models: {available_models}")
except Exception as e:
    print(f"Could not list models: {e}")
    available_models = []

model_names_to_try = [
    'gemini-flash-latest',
    'models/gemini-flash-latest',
]

model = None
model_name = None

for name in model_names_to_try:
    try:
        # Remove 'models/' prefix if present in the name list but try both formats
        test_name = name.replace('models/', '') if name.startswith('models/') else name
        model = genai.GenerativeModel(test_name)
        # Test if model works by checking if it can be instantiated
        model_name = test_name
        print(f"✓ Successfully initialized model: {test_name}")
        break
    except Exception as e:
        print(f"  ✗ Failed to initialize {name}: {e}")
        continue

if model is None:
    # If none worked, try to use the first available model from the list
    if available_models:
        try:
            # Extract model name from full path (e.g., "models/gemini-1.5-pro" -> "gemini-1.5-pro")
            first_model = available_models[0].split('/')[-1]
            model = genai.GenerativeModel(first_model)
            model_name = first_model
            print(f"✓ Using first available model: {first_model}")
        except Exception as e:
            raise ValueError(f"Could not initialize any Gemini model. Error: {e}")
    else:
        raise ValueError(
            "Could not find any available Gemini models. "
            "Please check your API key and ensure you have access to Gemini models."
        )

print(f"\n✓ Gemini API configured successfully!")
print(f"Using model: {model_name}")



Checking available Gemini models...
Available models: ['models/gemini-2.5-flash', 'models/gemini-2.5-pro', 'models/gemini-2.0-flash-exp', 'models/gemini-2.0-flash', 'models/gemini-2.0-flash-001', 'models/gemini-2.0-flash-lite-001', 'models/gemini-2.0-flash-lite', 'models/gemini-2.0-flash-lite-preview-02-05', 'models/gemini-2.0-flash-lite-preview', 'models/gemini-exp-1206', 'models/gemini-2.5-flash-preview-tts', 'models/gemini-2.5-pro-preview-tts', 'models/gemma-3-1b-it', 'models/gemma-3-4b-it', 'models/gemma-3-12b-it', 'models/gemma-3-27b-it', 'models/gemma-3n-e4b-it', 'models/gemma-3n-e2b-it', 'models/gemini-flash-latest', 'models/gemini-flash-lite-latest', 'models/gemini-pro-latest', 'models/gemini-2.5-flash-lite', 'models/gemini-2.5-flash-image-preview', 'models/gemini-2.5-flash-image', 'models/gemini-2.5-flash-preview-09-2025', 'models/gemini-2.5-flash-lite-preview-09-2025', 'models/gemini-3-pro-preview', 'models/gemini-3-pro-image-preview', 'models/nano-banana-pro-preview', 'model

## Convert Event Log to Text Format

Convert the PM4Py EventLog to a structured text format that Gemini can analyze.



In [24]:
def event_log_to_text(event_log) -> str:
    """Convert PM4Py EventLog to a readable text format for Gemini."""
    if not event_log or len(event_log) == 0:
        return "No traces found in event log."
    
    text_parts = []
    text_parts.append("EVENT LOG DATA")
    text_parts.append("=" * 60)
    text_parts.append(f"\nTotal traces: {len(event_log)}")
    total_events = sum(len(trace) for trace in event_log)
    text_parts.append(f"Total events: {total_events}\n")
    
    # Extract all unique activities
    all_activities = set()
    for trace in event_log:
        for event in trace:
            activity = event.get("concept:name", "unknown")
            all_activities.add(activity)
    
    text_parts.append(f"Unique activities: {', '.join(sorted(all_activities))}\n")
    text_parts.append("TRACES:")
    text_parts.append("-" * 60)
    
    for i, trace in enumerate(event_log):
        case_id = trace.attributes.get("concept:name", f"trace_{i}")
        activities = [event.get("concept:name", "unknown") for event in trace]
        
        text_parts.append(f"\nTrace {i+1} (Case ID: {case_id}):")
        text_parts.append(f"  Activity sequence: {' -> '.join(activities)}")
        text_parts.append(f"  Number of events: {len(activities)}")
        
        # Add timestamps if available
        if trace and "time:timestamp" in trace[0]:
            timestamps = [event.get("time:timestamp", "") for event in trace]
            text_parts.append(f"  Timestamps: {timestamps}")
    
    # Calculate directly-follows graph
    text_parts.append("\n" + "-" * 60)
    text_parts.append("DIRECTLY-FOLLOWS RELATIONSHIPS:")
    text_parts.append("-" * 60)
    
    dfg = {}  # (source, target) -> count
    for trace in event_log:
        activities = [event.get("concept:name", "unknown") for event in trace]
        for i in range(len(activities) - 1):
            source = activities[i]
            target = activities[i + 1]
            dfg[(source, target)] = dfg.get((source, target), 0) + 1
    
    # Sort by frequency
    sorted_dfg = sorted(dfg.items(), key=lambda x: x[1], reverse=True)
    for (source, target), count in sorted_dfg:
        text_parts.append(f"  {source} -> {target} (occurs {count} time(s))")
    
    return "\n".join(text_parts)

# Convert event log to text
event_log_text = event_log_to_text(event_log)
print(event_log_text)



EVENT LOG DATA

Total traces: 3
Total events: 9

Unique activities: end, process, review, start

TRACES:
------------------------------------------------------------

Trace 1 (Case ID: case1):
  Activity sequence: start -> process -> end
  Number of events: 3
  Timestamps: [Timestamp('2024-01-01 10:00:00+0000', tz='UTC'), Timestamp('2024-01-01 10:05:00+0000', tz='UTC'), Timestamp('2024-01-01 10:10:00+0000', tz='UTC')]

Trace 2 (Case ID: case2):
  Activity sequence: start -> end
  Number of events: 2
  Timestamps: [Timestamp('2024-01-01 11:00:00+0000', tz='UTC'), Timestamp('2024-01-01 11:05:00+0000', tz='UTC')]

Trace 3 (Case ID: case3):
  Activity sequence: start -> process -> review -> end
  Number of events: 4
  Timestamps: [Timestamp('2024-01-01 12:00:00+0000', tz='UTC'), Timestamp('2024-01-01 12:05:00+0000', tz='UTC'), Timestamp('2024-01-01 12:10:00+0000', tz='UTC'), Timestamp('2024-01-01 12:15:00+0000', tz='UTC')]

------------------------------------------------------------
DIREC

## Process Mining with Gemini

Use Gemini AI to analyze the event log and discover the process model.



In [25]:
# Construct the prompt for Gemini
prompt = f"""You are a process mining expert. Analyze the following event log data and discover the underlying business process model.

{event_log_text}

Based on this event log, please provide:

1. **Process Model Description**: Describe the discovered process model in BPMN-like terms, including:
   - The main activities/nodes
   - The flow relationships between activities
   - Any decision points (gateways) or parallel paths
   - Start and end nodes

2. **Process Flow**: Describe the typical process flow(s) observed in the data

3. **Key Patterns**: Identify any interesting patterns, such as:
   - Sequential flows
   - Optional activities
   - Parallel activities
   - Loops or cycles

4. **Graphviz DOT Format**: Provide the process model in Graphviz DOT format that can be visualized. 
   Use this format:
   ```
   digraph ProcessModel {{
       // Start node
       Start [shape=ellipse, style=filled, fillcolor=green];
       
       // Activities (rectangles)
       ActivityName [shape=box, label="Activity Name"];
       
       // End node
       End [shape=ellipse, style=filled, fillcolor=red];
       
       // Flows (edges)
       Start -> ActivityName;
       ActivityName -> End;
       
       // Gateways (diamonds) - use XOR for exclusive choice, AND for parallel
       GatewayName [shape=diamond, label="XOR"];
       ActivityName -> GatewayName;
       GatewayName -> End [label="Path 1"];
       GatewayName -> AnotherActivity [label="Path 2"];
   }}
   ```
   
   IMPORTANT: Include the complete DOT code in a code block marked with ```dot or ```graphviz

5. **Mermaid Format** (alternative): Also provide the process model in Mermaid flowchart format:
   ```
   flowchart TD
       Start([Start]) --> Activity1[Activity Name]
       Activity1 --> Gateway1{{XOR Gateway}}
       Gateway1 -->|Path 1| End1([End])
       Gateway1 -->|Path 2| Activity2[Another Activity]
       Activity2 --> End2([End])
   ```
   
   Include the complete Mermaid code in a code block marked with ```mermaid

Format your response with clear sections and include both DOT and Mermaid code blocks for visualization.
"""

print("Sending request to Gemini API...")
print("=" * 60)

try:
    response = model.generate_content(prompt)
    gemini_response = response.text
    print("✓ Gemini API response received!")
    print("\n" + "=" * 60)
    print("GEMINI RESPONSE:")
    print("=" * 60)
    print(gemini_response)
except Exception as e:
    print(f"✗ Error calling Gemini API: {e}")
    gemini_response = None
    raise



Sending request to Gemini API...
✓ Gemini API response received!

GEMINI RESPONSE:
This analysis applies techniques commonly used in process mining, specifically Alpha Miner or Heuristic Miner approaches, to construct a process model based on the observed Directly-Follows Relationships (DFR) and trace sequences.

---

## 1. Process Model Description

The discovered process model is characterized by sequential flow with two points of exclusive choice (XOR Gateways), indicating optional pathways for processing and reviewing.

| Element | Description |
| :--- | :--- |
| **Start/End Nodes** | The process begins with `start` and concludes with `end`. |
| **Activities** | `process` and `review`. |
| **Flow Relationships** | `start` flows to a decision point. `process` flows to a second decision point. `review` flows directly to `end`. |
| **Gateways (Decision Points)** | **XOR Split 1 (Initial Choice):** Immediately after `start`. The case either skips all processing (`start -> end`) or proc

## Parse and Analyze Gemini Response

Extract structured information from Gemini's response.



In [ ]:
import re

def extract_code_block(text, language="dot"):
    """Extract code block from text by language marker."""
    # Try different patterns
    patterns = [
        rf"```{language}\s*\n(.*?)```",
        rf"```{language}\n(.*?)```",
        rf"```\s*{language}\s*\n(.*?)```",
    ]
    
    for pattern in patterns:
        match = re.search(pattern, text, re.DOTALL)
        if match:
            return match.group(1).strip()
    
    # Also try without language marker
    match = re.search(r"```\n(.*?)```", text, re.DOTALL)
    if match:
        code = match.group(1).strip()
        # Check if it looks like DOT/Mermaid code
        if "digraph" in code or "flowchart" in code or "graph" in code:
            return code
    
    return None

if gemini_response:
    print("Gemini's Process Model Analysis:")
    print("=" * 60)
    print(gemini_response)
    print("=" * 60)
    
    # Extract DOT and Mermaid code
    dot_code = extract_code_block(gemini_response, "dot") or extract_code_block(gemini_response, "graphviz")
    mermaid_code = extract_code_block(gemini_response, "mermaid")
    
    print("\n" + "=" * 60)
    print("EXTRACTED VISUALIZATION CODE")
    print("=" * 60)
    
    if dot_code:
        print("\n✓ Found DOT/Graphviz code:")
        print("-" * 60)
        print(dot_code[:500] + ("..." if len(dot_code) > 500 else ""))
    else:
        print("\n⚠ No DOT code found in response")
    
    if mermaid_code:
        print("\n✓ Found Mermaid code:")
        print("-" * 60)
        print(mermaid_code[:500] + ("..." if len(mermaid_code) > 500 else ""))
    else:
        print("\n⚠ No Mermaid code found in response")
    
    # Save response to file
    results_dir = project_root / "results"
    results_dir.mkdir(exist_ok=True)
    
    gemini_output_file = results_dir / "gemini_process_model.txt"
    with open(gemini_output_file, "w", encoding="utf-8") as f:
        f.write("GEMINI PROCESS MINING RESULTS\n")
        f.write("=" * 60 + "\n\n")
        f.write("EVENT LOG SUMMARY\n")
        f.write("-" * 60 + "\n")
        f.write(f"Traces: {len(event_log)}\n")
        f.write(f"Total events: {sum(len(trace) for trace in event_log)}\n\n")
        f.write("GEMINI ANALYSIS\n")
        f.write("-" * 60 + "\n\n")
        f.write(gemini_response)
    
    print(f"\n✓ Saved Gemini response to: {gemini_output_file}")
    
    # Store extracted codes for visualization
    extracted_dot = dot_code
    extracted_mermaid = mermaid_code
else:
    print("⚠ No response from Gemini to parse.")
    extracted_dot = None
    extracted_mermaid = None



Gemini's Process Model Analysis:
This analysis uses the provided event log data to discover the underlying business process model, applying techniques common in process mining, such as the alpha miner or transition systems derivation.

---

## 1. Process Model Description

The discovered process model is characterized by a sequential flow with two distinct points of choice (gateways), resulting in three possible execution paths.

**Main Activities and Flow:**

1.  The process **must** begin with the **start** event.
2.  Following **start**, there is a decision point (XOR Split).
    *   The process can immediately conclude at **end** (Trace 2).
    *   The process can proceed to the core activity **process** (Traces 1 and 3).
3.  The activity **process** is executed only if the initial decision routes the case there.
4.  Following **process**, there is a second decision point (XOR Split).
    *   The case can immediately conclude at **end** (Trace 1).
    *   The case can proceed to **

## Visualize Process Model

Generate visual diagram from Gemini's structured output (DOT or Mermaid format).



In [ ]:
# Try to visualize using Graphviz (DOT) or Mermaid
results_dir = project_root / "results"
results_dir.mkdir(exist_ok=True)

diagram_generated = False

# Method 1: Try Graphviz DOT format
if extracted_dot:
    try:
        import graphviz
        
        # Create graph from DOT code
        graph = graphviz.Source(extracted_dot)
        
        # Save as PNG
        png_path = results_dir / "gemini_process_model.png"
        graph.render(str(png_path).replace('.png', ''), format='png', cleanup=True)
        
        # The render method creates .png file, so update path if needed
        actual_png = results_dir / f"{png_path.stem}.png"
        if actual_png.exists():
            print(f"✓ Generated diagram from DOT code: {actual_png}")
            diagram_generated = True
            diagram_path = actual_png
        else:
            print("⚠ DOT code processed but PNG file not found")
            
    except ImportError:
        print("⚠ graphviz not installed. Install with: pip install graphviz")
        print("   Also install Graphviz system package: https://graphviz.org/download/")
    except Exception as e:
        print(f"⚠ Failed to generate diagram from DOT: {e}")

# Method 2: Try Mermaid format (requires mermaid.ink API or local rendering)
if not diagram_generated and extracted_mermaid:
    try:
        # Option: Use mermaid.ink API to generate image
        import urllib.parse
        import urllib.request
        
        encoded = urllib.parse.quote(extracted_mermaid)
        mermaid_url = f"https://mermaid.ink/img/{encoded}"
        
        # Download the image
        png_path = results_dir / "gemini_process_model_mermaid.png"
        urllib.request.urlretrieve(mermaid_url, str(png_path))
        
        print(f"✓ Generated diagram from Mermaid code: {png_path}")
        diagram_generated = True
        diagram_path = png_path
        
    except Exception as e:
        print(f"⚠ Failed to generate diagram from Mermaid: {e}")
        print("   Note: Mermaid visualization requires internet connection for mermaid.ink API")

# Method 3: Fallback - try using pm4py to create a simple visualization
if not diagram_generated:
    print("\n⚠ Could not generate diagram from structured code.")
    print("   You can manually copy the DOT or Mermaid code to visualize it:")
    print("   - DOT: Use online Graphviz editor (https://dreampuf.github.io/GraphvizOnline/)")
    print("   - Mermaid: Use Mermaid Live Editor (https://mermaid.live/)")
    
    if extracted_dot:
        print("\nDOT Code:")
        print("-" * 60)
        print(extracted_dot)
    
    if extracted_mermaid:
        print("\nMermaid Code:")
        print("-" * 60)
        print(extracted_mermaid)



In [ ]:
# Display the generated diagram if available
if diagram_generated and 'diagram_path' in locals():
    try:
        from IPython.display import Image, display
        
        if diagram_path.exists():
            print("\n" + "=" * 60)
            print("GENERATED PROCESS MODEL DIAGRAM")
            print("=" * 60)
            display(Image(str(diagram_path)))
        else:
            print(f"⚠ Diagram file not found at: {diagram_path}")
    except Exception as e:
        print(f"⚠ Could not display diagram: {e}")
        print(f"   Diagram saved at: {diagram_path}")
else:
    print("\n💡 Tip: To visualize the process model, you can:")
    print("   1. Copy the DOT/Mermaid code from above")
    print("   2. Use online tools like GraphvizOnline or Mermaid Live Editor")
    print("   3. Or install graphviz locally: pip install graphviz")



## Export Results

Export the processed event log snapshot.



In [19]:
results_dir = project_root / "results"
results_dir.mkdir(exist_ok=True)

from stream_mining.io_xes import Log, Trace

window_log = Log(traces=[])
case_events = {}

for event in window.iter_events():
    if event.case_id not in case_events:
        case_events[event.case_id] = []
    case_events[event.case_id].append(event)

for case_id, events in case_events.items():
    trace = Trace(case_id=case_id, events=sorted(events, key=lambda e: e.timestamp))
    window_log.traces.append(trace)

xes_output = results_dir / "stream_snapshot_gemini.xes"
export_xes(window_log, str(xes_output))
print(f"✓ Exported XES snapshot: {xes_output}")



exporting log, completed traces ::   0%|          | 0/3 [00:00<?, ?it/s]

✓ Exported XES snapshot: C:\github\process-mining\results\stream_snapshot_gemini.xes


## Pipeline Summary

Display statistics and summary of the Gemini-based process mining pipeline.



In [20]:
print("=" * 60)
print("PIPELINE SUMMARY (GEMINI AI)")
print("=" * 60)

print(f"\n📊 Input Data:")
print(f"  - Original traces: {len(log.traces)}")
original_events = sum(len(trace.events) for trace in log.traces)
print(f"  - Original events: {original_events}")

print(f"\n🪟 Sliding Window:")
print(f"  - Current size: {len(window)} / {window.max_events}")
window_stats = window.stats()
if window_stats.get('oldest_timestamp'):
    window_span = window_stats['newest_timestamp'] - window_stats['oldest_timestamp']
    print(f"  - Time span: {window_span}")

print(f"\n📉 Decay List:")
decay_stats = decay.stats(reference_time=now)
print(f"  - Total events: {decay_stats['size']}")
if decay_stats.get('min_weight') is not None:
    print(f"  - Weight range: {decay_stats['min_weight']:.4f} - {decay_stats['max_weight']:.4f}")

print(f"\n🔧 Prepared Dataset:")
print(f"  - Traces for mining: {len(event_log)}")
mining_events = sum(len(trace) for trace in event_log)
print(f"  - Events for mining: {mining_events}")

print(f"\n🤖 AI Model:")
print(f"  - Provider: Google Gemini")
try:
    print(f"  - Model: {model_name}")
except NameError:
    print(f"  - Model: N/A (run previous cells first)")
print(f"  - Status: {'✓ Analysis completed' if gemini_response else '✗ Analysis failed'}")

print(f"\n💾 Output Files:")
print(f"  - XES snapshot: {xes_output}")
if gemini_response:
    gemini_output_file = results_dir / "gemini_process_model.txt"
    print(f"  - Gemini analysis: {gemini_output_file}")

print("\n" + "=" * 60)
print("\nNOTE: This notebook uses Gemini AI for process discovery instead of")
print("traditional algorithms like Heuristics Miner. The AI analyzes the")
print("event log patterns and provides a natural language description of")
print("the discovered process model.")



PIPELINE SUMMARY (GEMINI AI)

📊 Input Data:
  - Original traces: 3
  - Original events: 9

🪟 Sliding Window:
  - Current size: 9 / 50
  - Time span: 0 days 02:15:00

📉 Decay List:
  - Total events: 0

🔧 Prepared Dataset:
  - Traces for mining: 3
  - Events for mining: 9

🤖 AI Model:
  - Provider: Google Gemini
  - Model: gemini-flash-latest
  - Status: ✓ Analysis completed

💾 Output Files:
  - XES snapshot: C:\github\process-mining\results\stream_snapshot_gemini.xes
  - Gemini analysis: C:\github\process-mining\results\gemini_process_model.txt


NOTE: This notebook uses Gemini AI for process discovery instead of
traditional algorithms like Heuristics Miner. The AI analyzes the
event log patterns and provides a natural language description of
the discovered process model.
